# Sesión de clase 14 · Ejercicio 3: Desarrollo de un MCP server para la Tienda Virtual

**Contexto:** ya existen un RAG semántico (Ejercicio 1), una ontología con RAG determinista
(Ejercicio 2) y la API REST de la tienda (scrapeada en las primeras sesiones). Cada una vive en un
notebook distinto y para usarlas desde un asistente habría que escribir código de integración a mano,
una vez por cada asistente. **MCP** (*Model Context Protocol*) es un protocolo abierto que estandariza
cómo una aplicación con LLM descubre y usa herramientas y datos externos.

**Se construye:** `mcp_tienda/herramientas.py` (la lógica) y `mcp_tienda/server.py` (el MCP server),
y se prueban con un cliente MCP en Python.

**Requisitos:** haber ejecutado los ejercicios 1 y 2 (crean `../bdv` y `../ontologia`) y tener la
Tienda Virtual levantada en `http://localhost:3000`.

**Etapas:** conceptos · lógica de las herramientas · server con tools, resources y prompts · cliente
MCP · **comparación: sin MCP vs. con MCP** (stock en vivo).

## 1. Conceptos

### 1.1 El problema que resuelve

Sin un estándar, cada aplicación (un chat, un IDE, un agente) necesita su propio conector para cada
fuente de datos: *N aplicaciones × M fuentes* integraciones. Con MCP, cada fuente se expone **una vez**
como server y cualquier aplicación compatible la usa: *N + M*.

```mermaid
graph LR
  subgraph Host["Host (app con LLM)"]
    LLM[LLM] --- C1[Cliente MCP]
    LLM --- C2[Cliente MCP]
  end
  C1 -- "JSON-RPC (stdio)" --> S1["MCP server<br/>tienda-virtual"]
  C2 -- "JSON-RPC (HTTP)" --> S2["Otro MCP server"]
  S1 --> B[(bdv · Chroma)]
  S1 --> O[(ontología · RDF)]
  S1 --> A[API REST<br/>localhost:3000]
```

| Rol | Qué hace | Aquí |
|---|---|---|
| **Host** | aplicación donde vive el LLM y conversa el usuario | este notebook (y el agente del notebook 04) |
| **Cliente** | mantiene la conexión 1:1 con un server | `mcp.Client` |
| **Server** | expone capacidades de una fuente | `mcp_tienda/server.py` |

### 1.2 Las tres primitivas

| Primitiva | Quién la controla | Para qué | En la tienda |
|---|---|---|---|
| **Tools** | el modelo decide llamarlas | acciones y consultas con parámetros | `buscar_productos`, `consultar_ontologia`, `stock_en_vivo`… |
| **Resources** | la aplicación las lee | datos de contexto de solo lectura, identificados por URI | `ontologia://esquema`, `tienda://categorias` |
| **Prompts** | el usuario los elige | plantillas de instrucciones reutilizables | `analista_de_producto` |

### 1.3 El protocolo

Los mensajes son **JSON-RPC 2.0**. El transporte puede ser **stdio** (el cliente lanza el server como
subproceso y se hablan por stdin/stdout) o **HTTP** (server remoto). El ciclo es:

```mermaid
sequenceDiagram
  participant C as Cliente
  participant S as Server
  C->>S: initialize / discover
  S-->>C: capacidades (tools, resources, prompts)
  C->>S: tools/list
  S-->>C: nombre + descripción + inputSchema de cada tool
  C->>S: tools/call {"name": "stock_en_vivo", "arguments": {"producto": "DJI Air 3"}}
  S-->>C: resultado (texto / JSON)
```

Una llamada a una tool viaja así:

```json
{"jsonrpc": "2.0", "id": 7, "method": "tools/call",
 "params": {"name": "stock_en_vivo", "arguments": {"producto": "DJI Air 3"}}}
```

## Configuración

In [1]:
import os, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"; os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
import json, re, sys, time
from pathlib import Path

import pandas as pd
import requests
import torch
import transformers; transformers.logging.set_verbosity_error()
from transformers import AutoTokenizer, AutoModelForCausalLM

TIENDA = "http://localhost:3000"
API_KEY = "sk_demo_000000000000000000000000000000"
MODELO_LLM = "Qwen/Qwen2.5-0.5B-Instruct"
Path("../mcp_tienda").mkdir(exist_ok=True)
print("Tienda:", requests.get(f"{TIENDA}/api/health", timeout=5).json())

Tienda: {'status': 'ok', 'service': 'tienda-virtual-api', 'time': '2026-11-24T22:21:07.571Z'}


## 2. Paso 1 · La lógica de las herramientas (`herramientas.py`)

Primero se escribe la lógica **sin nada de MCP**: funciones Python normales que reutilizan lo
construido en los ejercicios anteriores. Separarla del protocolo permite probarla sola y cambiar de
transporte o de framework sin tocarla.

### 2.1 Configuración y recursos compartidos

Rutas, URL de la tienda y la API key (configurables por variables de entorno). Los modelos, la base
vectorial y el grafo se cargan **la primera vez que se usan** (`lru_cache`), para que el server
arranque rápido.

In [2]:
%%writefile ../mcp_tienda/herramientas.py
"""Lógica de las herramientas de la Tienda Virtual (sin nada de MCP).

Empaqueta lo construido en los ejercicios 1 y 2 para que el MCP server lo exponga:
- RAG semántico sobre la base vectorial `bdv/` (Ejercicio 1)
- RAG determinista sobre la ontología `ontologia/` (Ejercicio 2)
- Datos en vivo desde la API REST de la Tienda Virtual

Todo se carga de forma perezosa (la primera vez que se usa) para que el server arranque rápido.
"""

from __future__ import annotations

import json
import os
import re
import unicodedata
from datetime import datetime
from functools import lru_cache
from pathlib import Path

import requests

RAIZ = Path(__file__).resolve().parents[1]
BDV = RAIZ / "bdv"
ONTO = RAIZ / "ontologia"
MODELO_EMB = "paraphrase-multilingual-MiniLM-L12-v2"
TIENDA_URL = os.environ.get("TIENDA_URL", "http://localhost:3000")
API_KEY = os.environ.get("TIENDA_API_KEY", "sk_demo_000000000000000000000000000000")
UMBRAL = 0.55
PREFIJOS = ("PREFIX tv: <http://tienda.local/ontologia#>\n"
            "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n")


# ---------------------------------------------------------------- recursos compartidos
@lru_cache
def _modelo_emb():
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(MODELO_EMB)


@lru_cache
def _colecciones():
    import chromadb
    from chromadb.utils import embedding_functions
    fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=MODELO_EMB)
    cliente = chromadb.PersistentClient(path=str(BDV))
    return (cliente.get_collection("productos", embedding_function=fn),
            cliente.get_collection("resenas", embedding_function=fn))


@lru_cache
def _grafo():
    from rdflib import Graph
    return Graph().parse(ONTO / "tienda_poblada.ttl")


def _sparql(consulta: str, **parametros) -> list[dict]:
    from rdflib import Literal
    enlaces = {k: Literal(v) for k, v in parametros.items()}
    r = _grafo().query(PREFIJOS + consulta, initBindings=enlaces)
    return [{str(var): (fila[i].toPython() if fila[i] is not None else None)
             for i, var in enumerate(r.vars)} for fila in r]


@lru_cache
def _intenciones():
    intenciones = json.loads((ONTO / "intenciones.json").read_text())
    ejemplos = [(i["nombre"], e) for i in intenciones for e in i["ejemplos"]]
    emb = _modelo_emb().encode([e for _, e in ejemplos], normalize_embeddings=True)
    return {i["nombre"]: i for i in intenciones}, [n for n, _ in ejemplos], emb


@lru_cache
def _diccionario():
    etiqueta = lambda clase, prop="rdfs:label": [
        f["l"] for f in _sparql(f"SELECT DISTINCT ?l WHERE {{ ?x a tv:{clase} ; {prop} ?l }}")]
    return {"producto": etiqueta("Producto", "tv:nombre"), "categoria": etiqueta("Categoria"),
            "subcategoria": etiqueta("Subcategoria"), "pais": etiqueta("Pais")}


Overwriting ../mcp_tienda/herramientas.py


### 2.2 Utilidades de texto

La misma búsqueda de entidades del Ejercicio 2: encuentra el nombre oficial de un producto,
categoría o país dentro de un texto libre. `resolver_producto` traduce "el DJI Air 3" al `id` que
necesita la API.

In [3]:
%%writefile -a ../mcp_tienda/herramientas.py


# ---------------------------------------------------------------- utilidades de texto
def _norm(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    return re.sub(r"[^a-z0-9 ]", " ", "".join(c for c in texto if not unicodedata.combining(c)))


def buscar_entidad(texto: str, candidatos: list[str]) -> str | None:
    """Candidato con mayor fracción de sus palabras presentes en el texto (≥ 0.6)."""
    palabras = set(_norm(texto).split())
    palabras |= {w[:-2] for w in palabras if w.endswith("es")} | {w[:-1] for w in palabras if w.endswith("s")}
    mejor, mejor_puntaje = None, 0.0
    for c in candidatos:
        tokens = [t for t in _norm(c).split() if t not in {"y", "de"}]
        puntaje = sum(t in palabras for t in tokens) / len(tokens)
        if puntaje > mejor_puntaje or (puntaje == mejor_puntaje and mejor
                                       and len(tokens) > len(_norm(mejor).split())):
            mejor, mejor_puntaje = c, puntaje
    return mejor if mejor_puntaje >= 0.6 else None


def resolver_producto(texto: str) -> dict | None:
    """Nombre aproximado → ficha del producto en la ontología (id, código, nombre)."""
    nombre = buscar_entidad(texto, _diccionario()["producto"])
    if nombre is None:
        return None
    fila = _sparql("SELECT ?p ?codigo WHERE { ?p tv:nombre ?nombre ; tv:codigo ?codigo }", nombre=nombre)[0]
    return {"id": int(str(fila["p"]).rsplit("_", 1)[-1]), "codigo": fila["codigo"], "nombre": nombre}


Appending to ../mcp_tienda/herramientas.py


### 2.3 RAG semántico (Ejercicio 1)

Consultas a ChromaDB con filtros opcionales por metadatos. Devuelven **datos estructurados** (no
texto redactado): quien llame a la herramienta decide cómo presentarlos.

In [4]:
%%writefile -a ../mcp_tienda/herramientas.py


# ---------------------------------------------------------------- herramientas: RAG
def buscar_productos(consulta: str, categoria: str | None = None,
                     precio_max: float | None = None, k: int = 5) -> list[dict]:
    filtros = []
    if categoria:
        cat = buscar_entidad(categoria, _diccionario()["categoria"])
        if cat:
            filtros.append({"categoria": cat})
    if precio_max:
        filtros.append({"precio": {"$lte": float(precio_max)}})
    where = None if not filtros else filtros[0] if len(filtros) == 1 else {"$and": filtros}
    col_productos, _ = _colecciones()
    r = col_productos.query(query_texts=[consulta], n_results=k, where=where)
    return [{"nombre": m["nombre"], "categoria": m["categoria"], "subcategoria": m["subcategoria"],
             "precio": m["precio"], "calificacion": m["calificacion"],
             "similitud": round(1 - d, 3)}
            for m, d in zip(r["metadatas"][0], r["distances"][0])]


def buscar_resenas(consulta: str, solo_negativas: bool = False, k: int = 5) -> list[dict]:
    _, col_resenas = _colecciones()
    where = {"calificacion": {"$lte": 2}} if solo_negativas else None
    r = col_resenas.query(query_texts=[consulta], n_results=k, where=where)
    return [{"producto": m["producto"], "pais": m["pais"], "calificacion": m["calificacion"],
             "fecha": m["fecha"], "texto": doc.split(": ", 1)[-1]}
            for m, doc in zip(r["metadatas"][0], r["documents"][0])]


Appending to ../mcp_tienda/herramientas.py


### 2.4 RAG determinista (Ejercicio 2)

El router de intenciones, la extracción de parámetros y las plantillas SPARQL. El catálogo se lee de
`../ontologia/intenciones.json`, el mismo que se definió en el Ejercicio 2. Si la pregunta no
corresponde a una intención, la herramienta **lo dice** y sugiere qué otra usar.

In [5]:
%%writefile -a ../mcp_tienda/herramientas.py


# ---------------------------------------------------------------- herramienta: RAG determinista
def _formatear(intencion: dict, parametros: dict, filas: list[dict]) -> str:
    if not filas or all(v is None for v in filas[0].values()):
        return "No hay resultados para esa consulta en la ontología."
    texto = intencion["encabezado"].format(**{**parametros, **filas[0], "n": len(filas)})
    if intencion["fila"]:
        texto += " " + "; ".join(intencion["fila"].format(**f) for f in filas) + "."
    return texto


def clasificar(pregunta: str) -> tuple[str, float]:
    _, nombres, emb = _intenciones()
    sim = emb @ _modelo_emb().encode(pregunta, normalize_embeddings=True)
    mejor = {}
    for n, s in zip(nombres, sim):
        mejor[n] = max(mejor.get(n, -1.0), float(s))
    nombre = max(mejor, key=mejor.get)
    return nombre, mejor[nombre]


def consultar_ontologia(pregunta: str) -> dict:
    catalogo, _, _ = _intenciones()
    nombre, confianza = clasificar(pregunta)
    intencion = catalogo[nombre]
    numeros = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", pregunta.replace(",", ""))]
    parametros = {}
    for p in intencion["parametros"]:
        if p in ("umbral", "precio_max"):
            parametros[p] = int(numeros[-1]) if numeros else None
        else:
            parametros[p] = buscar_entidad(pregunta, _diccionario()[p])
    faltan = [p for p, v in parametros.items() if v is None]
    if confianza < UMBRAL or faltan or intencion["sparql"] is None:
        motivo = (f"confianza {confianza:.2f} < {UMBRAL}" if confianza < UMBRAL
                  else f"faltan parámetros: {faltan}" if faltan else "pregunta abierta")
        return {"determinista": False, "intencion": nombre, "confianza": round(confianza, 2),
                "motivo": motivo,
                "sugerencia": "Usa buscar_productos o buscar_resenas para esta pregunta."}
    filas = _sparql(intencion["sparql"], **parametros)
    return {"determinista": True, "intencion": nombre, "confianza": round(confianza, 2),
            "parametros": parametros, "sparql": intencion["sparql"],
            "respuesta": _formatear(intencion, parametros, filas)}


Appending to ../mcp_tienda/herramientas.py


### 2.5 Datos en vivo y resources

Consultas a la API REST de la tienda con la API key (como en la sesión de scraping de APIs). Si la
tienda no responde, se devuelve un error legible en lugar de una excepción.

In [6]:
%%writefile -a ../mcp_tienda/herramientas.py


# ---------------------------------------------------------------- herramientas: API en vivo
def _api(ruta: str, **params):
    try:
        r = requests.get(f"{TIENDA_URL}{ruta}", params=params, headers={"x-api-key": API_KEY}, timeout=5)
    except (requests.ConnectionError, requests.Timeout):
        raise RuntimeError(f"La Tienda Virtual no responde en {TIENDA_URL}. "
                           "Levántala con `npm run dev` en la carpeta tienda-virtual.") from None
    if r.status_code >= 400:
        raise RuntimeError(f"La API respondió {r.status_code}: {r.text[:200]}")
    return r.json()


def stock_en_vivo(producto: str) -> dict:
    ficha = resolver_producto(producto)
    if ficha is None:
        return {"error": f"No reconozco el producto '{producto}' en el catálogo."}
    try:
        datos = _api(f"/api/productos/{ficha['id']}")
    except RuntimeError as e:
        return {"error": str(e)}
    return {"producto": datos["nombre"], "codigo": datos["codigo"], "stock": datos["stock"],
            "precio": datos["precio"], "fuente": f"{TIENDA_URL}/api/productos/{ficha['id']}",
            "consultado": datetime.now().isoformat(timespec="seconds")}


def ordenes_cliente(cliente_id: int, max_ordenes: int = 5) -> dict:
    try:
        datos = _api("/api/ordenes", clienteId=int(cliente_id), pageSize=max_ordenes)
    except RuntimeError as e:
        return {"error": str(e)}
    return {"cliente_id": int(cliente_id), "total_ordenes": datos["pageInfo"]["total"],
            "ordenes": [{"numero": o["numero"], "fecha": o["fecha"], "total": o["total"],
                         "productos": [it["producto"]["nombre"] for it in o["items"]]}
                        for o in datos["items"]]}


# ---------------------------------------------------------------- recursos (solo lectura)
def esquema_ontologia() -> str:
    return (ONTO / "tienda.ttl").read_text()


def categorias() -> dict:
    filas = _sparql("SELECT ?categoria ?subcategoria WHERE { ?s a tv:Subcategoria ; rdfs:label ?subcategoria ; "
                    "tv:subcategoriaDe/rdfs:label ?categoria } ORDER BY ?categoria ?subcategoria")
    resultado: dict[str, list[str]] = {}
    for f in filas:
        resultado.setdefault(f["categoria"], []).append(f["subcategoria"])
    return resultado


Appending to ../mcp_tienda/herramientas.py


In [9]:
# La lógica se puede probar sin MCP
sys.path.insert(0, "../mcp_tienda")
import importlib, herramientas as h
importlib.reload(h)

print(h.consultar_ontologia("¿Cuál es el precio del Nintendo Switch OLED?")["respuesta"])
print(h.stock_en_vivo("DJI Air 3"))
pd.DataFrame(h.buscar_productos("audífonos para el avión", precio_max=1500))

El precio de Nintendo Switch OLED es S/ 1,599.00.
{'producto': 'DJI Air 3', 'codigo': 'PROD-067', 'stock': 4, 'precio': 6999, 'fuente': 'http://localhost:3000/api/productos/67', 'consultado': '2026-11-24T17:24:47'}


,nombre,categoria,subcategoria,precio,calificacion,similitud
0,Apple AirPods Pro 2da Generacion,Audio,Audifonos Inalambricos,999,4.0,0.466
1,Marshall Emberton II,Audio,Parlantes Portatiles,899,3.0,0.444
2,Sony SRS-XB13,Audio,Parlantes Portatiles,249,3.0,0.422
3,Xiaomi Redmi Note 13 Pro 256GB,Smartphones,Gama Media,999,2.8,0.375
4,Samsung Galaxy Buds2 Pro,Audio,Audifonos Inalambricos,799,3.3,0.371


## 3. Paso 2 · El MCP server (`server.py`)

Con el SDK oficial (`mcp`), un server es un objeto `MCPServer` y cada capacidad es una función
decorada:

- `@mcp.tool()` — el **nombre** de la función es el nombre de la tool, el **docstring** es la
  descripción que lee el LLM para decidir cuándo usarla y los **type hints** se convierten en el
  `inputSchema` (JSON Schema) que valida los argumentos.
- `@mcp.resource("uri://...")` — datos de solo lectura con una URI.
- `@mcp.prompt()` — una plantilla con parámetros.

`mcp.run()` atiende JSON-RPC por stdio. La descripción de cada tool es parte del diseño: es lo único
que el modelo sabe de ella.

In [10]:
%%writefile ../mcp_tienda/server.py
"""MCP server de la Tienda Virtual.

Expone como herramientas MCP el RAG semántico, el RAG determinista sobre la ontología y los datos
en vivo de la API REST. Se comunica por stdio (JSON-RPC sobre stdin/stdout):

    python mcp_tienda/server.py
"""

import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))   # para importar herramientas.py

from mcp.server.mcpserver import MCPServer

import herramientas as h

mcp = MCPServer(
    "tienda-virtual",
    instructions=("Herramientas de 'Mi Tienda Virtual'. Para conteos, promedios, rankings y precios usa "
                  "consultar_ontologia; para descripciones y opiniones usa buscar_productos o buscar_resenas; "
                  "para stock actual usa stock_en_vivo."),
)


# ------------------------------------------------------------------ tools
@mcp.tool()
def buscar_productos(consulta: str, categoria: str | None = None, precio_max: float | None = None) -> list[dict]:
    """Busca productos del catálogo por significado (RAG semántico). Útil para preguntas descriptivas
    como 'audífonos con cancelación de ruido'. Filtros opcionales: categoría y precio máximo en soles."""
    return h.buscar_productos(consulta, categoria, precio_max)


@mcp.tool()
def buscar_resenas(consulta: str, solo_negativas: bool = False) -> list[dict]:
    """Busca reseñas de entrega de clientes por significado. Útil para opiniones y quejas.
    Con solo_negativas=True devuelve solo reseñas de 1 o 2 estrellas."""
    return h.buscar_resenas(consulta, solo_negativas)


@mcp.tool()
def consultar_ontologia(pregunta: str) -> dict:
    """Responde de forma exacta y reproducible preguntas de precios, conteos, promedios, rankings y
    relaciones (ventas por país, reseñas negativas por país, productos en riesgo) con SPARQL sobre la
    ontología de la tienda. Devuelve la respuesta y la consulta usada como evidencia."""
    return h.consultar_ontologia(pregunta)


@mcp.tool()
def stock_en_vivo(producto: str) -> dict:
    """Consulta el stock y precio ACTUAL de un producto en la API de la Tienda Virtual."""
    return h.stock_en_vivo(producto)


@mcp.tool()
def ordenes_cliente(cliente_id: int) -> dict:
    """Devuelve las órdenes más recientes de un cliente (por su id) desde la API de la Tienda Virtual."""
    return h.ordenes_cliente(cliente_id)


# ------------------------------------------------------------------ resources
@mcp.resource("ontologia://esquema", mime_type="text/turtle")
def esquema() -> str:
    """TBox de la ontología (clases y propiedades) en Turtle."""
    return h.esquema_ontologia()


@mcp.resource("tienda://categorias", mime_type="application/json")
def categorias() -> str:
    """Categorías y subcategorías del catálogo."""
    return json.dumps(h.categorias(), ensure_ascii=False)


# ------------------------------------------------------------------ prompts
@mcp.prompt()
def analista_de_producto(producto: str) -> str:
    """Plantilla para analizar un producto combinando datos exactos, opiniones y stock en vivo."""
    return (f"Analiza el producto '{producto}' de Mi Tienda Virtual. Usa consultar_ontologia para su "
            f"precio, buscar_resenas para lo que opinan los clientes y stock_en_vivo para el stock actual. "
            f"Termina con una recomendación de una frase.")


if __name__ == "__main__":
    mcp.run()   # transporte stdio por defecto

Overwriting ../mcp_tienda/server.py


## 4. Paso 3 · Un cliente MCP

El cliente lanza `server.py` como subproceso (transporte stdio) y se comunica con él por JSON-RPC.
En un script bastaría con `async with Client(...) as cliente:`. En Jupyter la conexión debe quedar
abierta entre celdas, y cada celda corre en una tarea asyncio distinta, así que la conexión se
mantiene en una **tarea propia** que la abre y la cierra. Los mensajes de log del server van a
`../salidas/mcp_server.log`.

In [11]:
from mcp import Client, StdioServerParameters, stdio_client

import asyncio

async def abrir_cliente(parametros, log):
    # Abre un cliente MCP en una tarea propia y devuelve (cliente, función para cerrarlo)
    listo, cerrar = asyncio.get_running_loop().create_future(), asyncio.Event()
    async def mantener():
        try:
            async with Client(stdio_client(parametros, errlog=log)) as c:
                listo.set_result(c)
                await cerrar.wait()
        except BaseException as e:
            if not listo.done():
                listo.set_exception(e)
            else:
                raise
    tarea = asyncio.create_task(mantener())
    async def cerrar_cliente():
        cerrar.set(); await tarea
    return await listo, cerrar_cliente

parametros = StdioServerParameters(command=sys.executable, args=["../mcp_tienda/server.py"])
log_server = open("../salidas/mcp_server.log", "w")
cliente, cerrar_cliente = await abrir_cliente(parametros, log_server)

print("Server:", cliente.server_info.name, "·", cliente.instructions)

Server: tienda-virtual · Herramientas de 'Mi Tienda Virtual'. Para conteos, promedios, rankings y precios usa consultar_ontologia; para descripciones y opiniones usa buscar_productos o buscar_resenas; para stock actual usa stock_en_vivo.


### 4.1 Descubrimiento: `tools/list`

In [12]:
tools = (await cliente.list_tools()).tools
pd.DataFrame([{"tool": t.name, "descripción": t.description.split(".")[0],
               "parámetros": {k: v.get("type") or [a.get("type") for a in v.get("anyOf", [])]
                              for k, v in t.input_schema["properties"].items()},
               "requeridos": t.input_schema.get("required", [])} for t in tools])

,tool,descripción,parámetros,requeridos
0,buscar_productos,Busca productos del catálogo por significado (...,"{'consulta': 'string', 'categoria': ['string',...",[consulta]
1,buscar_resenas,Busca reseñas de entrega de clientes por signi...,"{'consulta': 'string', 'solo_negativas': 'bool...",[consulta]
2,consultar_ontologia,Responde de forma exacta y reproducible pregun...,{'pregunta': 'string'},[pregunta]
3,stock_en_vivo,Consulta el stock y precio ACTUAL de un produc...,{'producto': 'string'},[producto]
4,ordenes_cliente,Devuelve las órdenes más recientes de un clien...,{'cliente_id': 'integer'},[cliente_id]


In [13]:
# El JSON Schema completo que el server publica para una tool (generado desde los type hints)
print(json.dumps(next(t for t in tools if t.name == "buscar_productos").input_schema, indent=2, ensure_ascii=False))

{
  "type": "object",
  "properties": {
    "consulta": {
      "title": "Consulta",
      "type": "string"
    },
    "categoria": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Categoria"
    },
    "precio_max": {
      "anyOf": [
        {
          "type": "number"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Precio Max"
    }
  },
  "required": [
    "consulta"
  ],
  "title": "buscar_productosArguments"
}


### 4.2 Ejecución: `tools/call`

In [14]:
def resultado(r):
    """Contenido de un CallToolResult como objeto Python (JSON estructurado o texto)."""
    if r.structured_content is not None:
        return r.structured_content.get("result", r.structured_content)
    texto = "".join(getattr(c, "text", "") for c in r.content)
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return texto

r = await cliente.call_tool("consultar_ontologia", {"pregunta": "¿Qué país tiene el mayor monto total de ventas?"})
resultado(r)

{'determinista': True,
 'intencion': 'ventas_por_pais',
 'confianza': 0.9,
 'parametros': {},
 'sparql': 'SELECT ?pais (SUM(?t) AS ?ventas) WHERE { ?o a tv:Orden ; tv:total ?t ; tv:realizadaPor/tv:resideEn/rdfs:label ?pais } GROUP BY ?pais ORDER BY DESC(?ventas)',
 'respuesta': 'El país con el mayor monto de ventas es Peru, con S/ 2,011,695.86. Ranking: Peru: S/ 2,011,695.86; Chile: S/ 973,372.56; Colombia: S/ 884,112.64; Ecuador: S/ 721,735.20; Argentina: S/ 665,499.94; Uruguay: S/ 549,718.34; Mexico: S/ 418,566.06.'}

In [15]:
r = await cliente.call_tool("buscar_resenas", {"consulta": "el pedido llegó tarde", "solo_negativas": True})
pd.DataFrame(resultado(r))

,producto,pais,calificacion,fecha,texto
0,Insta360 X4,Mexico,2,2024-07-24,La Insta360 X4 llego con dos dias de retraso d...
1,Sennheiser Momentum 4,Chile,2,2024-10-23,El segundo pedido de audifonos Sennheiser lleg...
2,Samsung Family Hub Refrigeradora Inteligente,Peru,2,2024-12-30,La instalacion de este segundo pedido de Famil...
3,HP Pavilion 15 Intel Core i5,Chile,2,2024-02-27,La laptop HP llego con un dia de retraso adici...
4,Samsung The Frame 55 pulgadas,Peru,2,2024-01-15,El Samsung The Frame llego tres dias despues d...


In [16]:
r = await cliente.call_tool("ordenes_cliente", {"cliente_id": 35})
resultado(r)

{'cliente_id': 35,
 'total_ordenes': 2,
 'ordenes': [{'numero': 'OR-00229',
   'fecha': '2025-12-27',
   'total': 15446.2,
   'productos': ['Xiaomi Poco X6 Pro 512GB',
    'JBL Flip 6',
    'Sony PlayStation 5 Digital Edition',
    'Huawei Watch GT 4']},
  {'numero': 'OR-00073',
   'fecha': '2024-08-17',
   'total': 11061.32,
   'productos': ['Logitech C920 Webcam HD Pro',
    'Acer Aspire 5 Intel Core i5',
    'Samsung 43 pulgadas Crystal UHD']}]}

In [17]:
# El server valida los argumentos con el inputSchema: un tipo incorrecto se rechaza
r = await cliente.call_tool("ordenes_cliente", {"cliente_id": "treinta y cinco"})
print("is_error:", r.is_error, "→", r.content[0].text[:200])

is_error: True → Error executing tool ordenes_cliente: 1 validation error for ordenes_clienteArguments
cliente_id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value=


### 4.3 Resources y prompts

In [18]:
for recurso in (await cliente.list_resources()).resources:
    print(f"{recurso.uri}  ({recurso.mime_type})  {recurso.description}")
categorias = json.loads((await cliente.read_resource("tienda://categorias")).contents[0].text)
categorias

ontologia://esquema  (text/turtle)  TBox de la ontología (clases y propiedades) en Turtle.
tienda://categorias  (application/json)  Categorías y subcategorías del catálogo.


{'Accesorios y Componentes': ['Almacenamiento',
  'Cables y Adaptadores',
  'Cargadores y Baterias',
  'Componentes PC',
  'Perifericos'],
 'Audio': ['Audifonos Inalambricos', 'Parlantes Portatiles'],
 'Electrodomesticos Inteligentes': ['Asistentes Virtuales',
  'Cocina Inteligente',
  'Iluminacion Inteligente',
  'Linea Blanca Inteligente',
  'Robots Aspiradora',
  'Seguridad Inteligente'],
 'Fotografia y Video': ['Camaras DSLR',
  'Camaras Instantaneas',
  'Camaras Mirrorless',
  'Camaras de Accion',
  'Drones'],
 'Gaming y Consolas': ['Consolas', 'Perifericos Gaming'],
 'Laptops y Computadoras': ['Gaming',
  'Ultrabooks',
  'Uso General',
  'Uso Profesional'],
 'Smartphones': ['Gama Alta', 'Gama Basica', 'Gama Media'],
 'Televisores y Video': ['Televisores LED',
  'Televisores OLED',
  'Televisores QLED',
  'Televisores ULED'],
 'Wearables y Fitness': ['Bandas de Actividad',
  'Relojes Deportivos',
  'Smartwatches']}

In [19]:
print((await cliente.read_resource("ontologia://esquema")).contents[0].text[:600])

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix tv: <http://tienda.local/ontologia#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<http://tienda.local/ontologia> a owl:Ontology .

tv:ProductoEnRiesgo a owl:Class ;
    rdfs:comment "Producto con calificación < 3 y al menos una reseña de entrega negativa"@es ;
    rdfs:subClassOf tv:Producto .

tv:Testimonio a owl:Class ;
    rdfs:label "Testimonio"@es ;
    rdfs:comment "Comentario general sobre la tienda"@es ;
    rdfs:subClassOf tv:Resena .

tv:calificacion a owl:Datatyp


In [20]:
prompt = await cliente.get_prompt("analista_de_producto", {"producto": "DJI Air 3"})
print(prompt.messages[0].content.text)

Analiza el producto 'DJI Air 3' de Mi Tienda Virtual. Usa consultar_ontologia para su precio, buscar_resenas para lo que opinan los clientes y stock_en_vivo para el stock actual. Termina con una recomendación de una frase.


## 5. Comparación: sin MCP vs. con MCP

**Escenario:** se venden 3 drones DJI Air 3 *después* del scraping. En la tienda el stock baja de 4 a 1
(se simula con un `PUT` a la API). Luego se pregunta *"¿Cuánto stock queda ahora del DJI Air 3?"*:

| Enfoque | Cómo obtiene el dato |
|---|---|
| (a) LLM solo | de su entrenamiento (no lo sabe) |
| (b) Sin MCP: RAG semántico | documento indexado en `../bdv` durante el Ejercicio 1 |
| (c) Sin MCP: RAG determinista | ontología construida en el Ejercicio 2 |
| (d) Con MCP: LLM + tools descubiertas | el LLM recibe las tools de `tools/list`, elige una y el server consulta la API en vivo |

In [21]:
cabecera = {"x-api-key": API_KEY, "content-type": "application/json"}
producto_id = h.resolver_producto("DJI Air 3")["id"]
antes = requests.get(f"{TIENDA}/api/productos/{producto_id}", headers=cabecera).json()["stock"]
requests.put(f"{TIENDA}/api/productos/{producto_id}", headers=cabecera, json={"stock": 1}).raise_for_status()
ahora = requests.get(f"{TIENDA}/api/productos/{producto_id}", headers=cabecera).json()["stock"]
print(f"Stock del DJI Air 3: en el scraping = {antes} → en la tienda ahora = {ahora}")

Stock del DJI Air 3: en el scraping = 4 → en la tienda ahora = 1


In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_LLM)
llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, dtype=torch.float32); llm.eval()
SISTEMA = "Eres el asistente de 'Mi Tienda Virtual', una tienda peruana de tecnología. Responde en español, en pocas frases."

def generar(mensajes, tools=None, max_new_tokens=150):
    entrada = tokenizer.apply_chat_template(mensajes, tools=tools, add_generation_prompt=True,
                                            return_tensors="pt", return_dict=True)
    with torch.no_grad():
        salida = llm.generate(**entrada, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(salida[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True).strip()

pregunta = "¿Cuánto stock queda ahora del DJI Air 3?"

In [23]:
# (a) LLM solo
resp_llm = generar([{"role": "system", "content": SISTEMA}, {"role": "user", "content": pregunta}])

# (b) Sin MCP — RAG semántico: el documento indexado dice lo que había al momento del scraping
doc = h._colecciones()[0].get(where={"nombre": "DJI Air 3"})["documents"][0]
resp_rag = generar([{"role": "system", "content": SISTEMA},
                    {"role": "user", "content": f"Contexto:\n[1] {doc}\n\nCon la información del contexto, responde en una frase.\n\nPregunta: {pregunta}"}])

# (c) Sin MCP — RAG determinista sobre la ontología
resp_onto = h.consultar_ontologia(pregunta)["respuesta"]
print(resp_llm, "\n", resp_rag, "\n", resp_onto)

El stock del DJI Air 3 actualmente es de 10 unidades. 
 El DJI Air 3 tiene 4 unidades disponibles. 
 Según los datos scrapeados, DJI Air 3 tiene 4 unidades en stock.


**(d) Con MCP.** El LLM no conoce las tools de antemano: se las da el cliente a partir de
`tools/list`. Qwen 2.5 está entrenado para responder con un bloque `<tool_call>{json}</tool_call>`
cuando recibe tools en su *chat template*. El ciclo mínimo de un agente es:

1. `tools/list` → esquemas en formato de función para el LLM
2. el LLM genera la llamada → se parsea el JSON
3. `tools/call` en el server → resultado
4. el resultado vuelve al LLM como mensaje `tool` → respuesta final

In [24]:
def esquema_para_llm(t):
    props = {k: {"type": v.get("type") or next(a["type"] for a in v["anyOf"] if a["type"] != "null")}
             for k, v in t.input_schema["properties"].items()}
    return {"type": "function", "function": {"name": t.name, "description": t.description,
            "parameters": {"type": "object", "properties": props, "required": t.input_schema.get("required", [])}}}

tools_llm = [esquema_para_llm(t) for t in tools]                       # 1. descubrimiento
mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": pregunta}]
salida = generar(mensajes, tools=tools_llm)                            # 2. el LLM decide
print("LLM →", salida)

llamada = json.loads(re.search(r"<tool_call>\s*(\{.*\})\s*</tool_call>", salida, re.S).group(1))
datos = resultado(await cliente.call_tool(llamada["name"], llamada["arguments"]))   # 3. el server ejecuta
print("MCP →", datos)

mensajes += [{"role": "assistant", "content": salida},
             {"role": "tool", "content": json.dumps(datos, ensure_ascii=False)}]
resp_mcp = generar(mensajes)                                            # 4. respuesta final
print("LLM →", resp_mcp)

LLM → <tool_call>
{"name": "stock_en_vivo", "arguments": {"producto": "DJI Air 3"}}
</tool_call>
MCP → {'producto': 'DJI Air 3', 'codigo': 'PROD-067', 'stock': 1, 'precio': 6999, 'fuente': 'http://localhost:3000/api/productos/67', 'consultado': '2026-11-24T17:29:44'}
LLM → El DJI Air 3 está disponible con un stock de 1 unidad y se encuentra en la fuente oficial. Puedes consultar su precio a través de este link: <https://localhost:3000/api/productos/67>.


In [26]:
def stock_mencionado(texto):
    m = re.search(r"(\d+)\s*unidad", texto) or re.search(r"stock\D{0,25}(\d+)", texto)
    return int(m.group(1)) if m else None

comparacion = pd.DataFrame([
    {"enfoque": "(a) LLM solo", "respuesta": resp_llm, "fuente del dato": "entrenamiento del modelo"},
    {"enfoque": "(b) sin MCP · RAG semántico", "respuesta": resp_rag, "fuente del dato": "../bdv (scraping)"},
    {"enfoque": "(c) sin MCP · RAG determinista", "respuesta": resp_onto, "fuente del dato": "../ontologia (scraping)"},
    {"enfoque": "(d) con MCP · LLM + tools", "respuesta": resp_mcp, "fuente del dato": datos.get("fuente", "?")},
])
comparacion["stock reportado"] = comparacion.respuesta.map(stock_mencionado)
comparacion["¿correcto? (real = %d)" % ahora] = comparacion["stock reportado"] == ahora
comparacion.style.set_properties(subset=["respuesta"], **{"white-space": "pre-wrap", "text-align": "left"})

,enfoque,respuesta,fuente del dato,stock reportado,¿correcto? (real = 1)
0,(a) LLM solo,El stock del DJI Air 3 actualmente es de 10 unidades.,entrenamiento del modelo,10,False
1,(b) sin MCP · RAG semántico,El DJI Air 3 tiene 4 unidades disponibles.,../bdv (scraping),4,False
2,(c) sin MCP · RAG determinista,"Según los datos scrapeados, DJI Air 3 tiene 4 unidades en stock.",../ontologia (scraping),4,False
3,(d) con MCP · LLM + tools,El DJI Air 3 está disponible con un stock de 1 unidad y se encuentra en la fuente oficial. Puedes consultar su precio a través de este link: .,http://localhost:3000/api/productos/67,1,True


Más allá del dato en vivo, la diferencia está en **cómo se integra**:

| | Sin MCP (funciones en el notebook) | Con MCP |
|---|---|---|
| Datos | foto del momento del scraping | en vivo cuando la tool consulta la API |
| Descubrimiento | quien integra lee el código | `tools/list` publica nombre, descripción y esquema |
| Validación de argumentos | manual | automática contra el `inputSchema` |
| Reutilización | copiar el código en cada asistente | cualquier host MCP se conecta al mismo server |
| Aislamiento | el código corre dentro del asistente | proceso separado; la API key queda en el server |

El mismo server se puede registrar, sin cambiar una línea, en cualquier host compatible con MCP
(un IDE, una app de escritorio, un agente propio). Por ejemplo, la configuración típica de un host es:

```json
{"mcpServers": {"tienda-virtual": {"command": ".venv/bin/python", "args": ["mcp_tienda/server.py"]}}}
```

In [27]:
# Se restaura el stock original y se cierra la conexión con el server
requests.put(f"{TIENDA}/api/productos/{producto_id}", headers=cabecera, json={"stock": antes}).raise_for_status()
print("Stock restaurado:", requests.get(f"{TIENDA}/api/productos/{producto_id}", headers=cabecera).json()["stock"])
await cerrar_cliente()

Stock restaurado: 4


## Discusión

- **MCP separa *qué sabe hacer* una fuente de *quién la usa*.** La lógica (`herramientas.py`) no sabe
  nada del protocolo; el server (`server.py`) solo la publica; el cliente la descubre en tiempo de
  ejecución.
- **El docstring es la interfaz para el modelo.** Una descripción ambigua hace que el LLM elija mal
  la tool, aunque el código sea correcto.
- **Datos en vivo:** el RAG y la ontología son fotos del scraping; una tool que consulta la API
  responde con el estado actual. Ambas cosas se combinan: lo histórico y agregado desde la ontología,
  lo operativo desde la API.
- **Con Qwen 0.5B el ciclo funciona para preguntas simples**, pero el modelo puede elegir mal la tool
  o inventar argumentos cuando hay muchas tools. El notebook 04 mide eso y aplica estrategias para
  hacerlo fiable.
- **Seguridad:** el server tiene la API key con permisos de escritura; por eso solo expone tools de
  lectura. Una tool que modifica datos (como el `PUT` de esta demo) debería pedir confirmación al
  usuario.

## Reto opcional

1. Agregar la tool `comparar_productos(producto_a: str, producto_b: str) -> dict` que devuelva
   precio, stock en vivo y calificación de ambos (reutilizando `resolver_producto` y `stock_en_vivo`).
   Reiniciar el cliente y verificar que aparece en `tools/list`.
2. Agregar el resource `tienda://producto/{codigo}` (*resource template*) que devuelva la ficha de un
   producto desde la ontología.
3. Ejecutar el server con transporte HTTP (`mcp.run("streamable-http")`) y conectarse con
   `Client("http://localhost:8000/mcp")`.